# Homework 3 - DreamBooth
In this homework you will implement DreamBooth - a method of finetuning diffusion models.

Since scoring image generation is costly, the effectiveness of your model will be scored based on the quality of images in the reports you deliver. Please note
that you will be scored based on the report and the code so be sure to put work into both parts (that is, if the images for a task and your conclusions are not in the report you will get 0 points for the task). The report should be created using the latex template and be delivered as PDF.

You will run the finetuning using the dataset with dogs, which is delivered as a tar.gz archive. It contains 5 dog images of a specific breed.

You can get 10 points in total for solving the homework, the points will be split as follows:
*   Using Hugging Face [1pt]
*   Generate Images from Stable Diffusion [1pt]
*   Train DreamBooth [4pt]
*   Prior Preservation [1pt]
*   Mixing Models in Denoising [1pt]
*   Report Quality [2pt] (this will be multiplied by the sum of scores from other tasks and divided by 8)



In [1]:
import os
os.makedirs("data/dog", exist_ok=True)
os.makedirs("data/dog_images", exist_ok=True)
os.makedirs("data/output", exist_ok=True)
os.makedirs("data/output_images", exist_ok=True)

## Prerequisites
Before starting to solve this homework, you should read the following texts:
*   Paper introducing diffusion models to image generation: https://arxiv.org/pdf/2006.11239, most importantly section 2 and algorithms 1 and 2
*   Subsection "Departure to Latent Space" from section1 and section 3 of https://arxiv.org/pdf/2112.10752
*   DreamBooth paper https://arxiv.org/pdf/2208.12242



In [2]:
!pip install bitsandbytes
!pip install diffusers

In [3]:
import torch
import os

from PIL import Image
from diffusers import DiffusionPipeline, DDPMScheduler, DDIMScheduler,UNet2DModel
from diffusers import StableDiffusionPipeline
import random
import string


## Using Hugging Face [1pt]
Hugging Face is a company developing tools for convenient use of machine learning models. In this homework, we will be using their library diffusers, which has been created specifically for working with diffusion models.

Below there is an example code for image generation from a simple unconditional model.

In [4]:
%%script false --no-raise-error
use_cuda = torch.cuda.is_available()

def to_dev(x):
    return x.to('cuda') if use_cuda else x

# Initialize the schedulers
scheduler_ddpm = DDPMScheduler.from_pretrained("google/ddpm-ema-church-256")
scheduler_ddim = DDIMScheduler.from_pretrained("google/ddpm-ema-church-256")

# Initialize the model from a pretrained checkpoint
model = UNet2DModel.from_pretrained("google/ddpm-ema-church-256")
model = to_dev(model)

# Define a diffusion pipeline
pipeline = DiffusionPipeline.from_pretrained("google/ddpm-ema-church-256", unet=model, scheduler=scheduler_ddim)
pipeline = to_dev(pipeline)

# Generate an image using the pipeline
image = pipeline(num_inference_steps=100).images[0]
image

Generate 5 images using the model defined above, try each of the two schedulers defined above. Try different numbers of timesteps (200, 50 and 10) in denoising. The initial noise should be the same each time.
Describe the influence of number of steps on the generated images in both schedulers.
Compare the difference between the two schedulers.
Look at how both schedulers are defined mathematically.
Give an explanation (we don't expect a formal mathematical proof, just an intuitive understanding based on the formulas)
for the fact that for DDIM the objects generated for 10 steps are roughly the same as for 200, only with a worse quality,
while this is not the case for DDPM. Also explain the differences in quality between the schedulers. We expect the answer to be something along the lines of 'this part of the formula \<the part\> is responsible for \<something\> and hence \<something\>'.

In [5]:
%%script false --no-raise-error
### CODE START ###
from PIL import Image
import matplotlib.pyplot as plt
import contextlib, io
images = []

for sched in [scheduler_ddim, scheduler_ddpm]:
    for ts in [10, 50, 200]:
        pipeline = DiffusionPipeline.from_pretrained("google/ddpm-ema-church-256", unet=model, scheduler=sched)
        pipeline = to_dev(pipeline)

        for i in range(5):
            image = pipeline(num_inference_steps=ts).images[0]
            images += [image]

        fig, axes = plt.subplots(1, 5, figsize=(15, 3))
        for ax, img in zip(axes, images[-5:]):
            ax.imshow(img)
            ax.axis('off')

        plt.tight_layout()
        sched_str = f'{sched}'[:4]
        plt.savefig(f'{sched_str}_{ts}.png')
        plt.show()
### CODE END ###

## Generate Images from Stable Diffusion [1pt]
Stable diffusion is a group of pretrained latent diffusion model, that is conditioned on text.
Below, you will have to implement a function for generating images.
The suggested prompts are specified, but feel free to change them to demonstrate your findings better.
That is - below you are going to implement dreambooth.

In dream booth you finetune a trained diffusion model,
to generate specific images. In your case, you will finetune
it to generate a dog of specific breed, for a prompt "a photo of sks dog",
"sks" being a token than should not have any meaning before the finetuning.

Ideally, after the finetuning, a prompt "a photo of sks dog" should
result in photos of a dog from this new breed, while a prompt "a photo of a dog",
should result in an image distribution that is the same as before finetuning.
That is unfortunately not the case - all images of dogs will look more like this specific dog breed.

All photos of animals might also change, likely becoming the same color as the specific dog breed.
The further away a concept from a dog, the less this change should be visible - so for instance,
for the solution we have, images for "a sailboat at sea" are almost the same as before the finetuning,
while those for "a child playing" are changed, but the change in image diversity is not as drastic as for other animals.

Your solution might differ from ours, so demonstrating one prompt for which the results are virtually unchanged,
one prompt for which there is some change but can't be trivially detected by human eye and one prompt for which
the image diversity has been decreased might require use of different prompts than ours.

You shoud generate 3 images for each of 3 different prompts.
The function should save each image under a unique name.
The function should be deterministic.

In [6]:
def generate_images(prompt, num_images, model_path, save_path, device, batch_size, precision = 32):
    ### CODE START ###

    if precision == 16:
        pipe = StableDiffusionPipeline.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
        )
    else:
        pipe = StableDiffusionPipeline.from_pretrained(model_path)

    pipe = pipe.to(device)

    total_batches = (num_images + batch_size - 1) // batch_size
    image_count = 0

    for batch in tqdm(range(total_batches), desc=prompt, disable=False):
        current_batch_size = min(batch_size, num_images - image_count)
        images = pipe([prompt] * current_batch_size).images

        for img in images:
            filename = f"{save_path}/{prompt.replace(' ', '_')}_{image_count}.png"
            img.save(filename)
            image_count += 1

    ### CODE END ###

In [7]:
%%script false --no-raise-error
import torch
from diffusers import StableDiffusionPipeline
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
prompts = ["a sailboat at sea", "a child playing", "a photo of a cat", "a photo of a dog", "a photo of sks dog"]
num_images = 3
sample_batch_size = 3
model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
save_path = "~/Git/ML12/VR/hw3/data/example_images"
save_path = ""
device = 'cuda' if torch.cuda.is_available() else 'cpu'

for prompt in prompts:
  generate_images(prompt, num_images, model_path, save_path, device, sample_batch_size)

## Dataset components
There are no points for this part, but feel free to edit them as needed.
Please note that arguments `class_data_root`, `class_prompt`, `class_num` of `DreamBoothDataset`
will only be used in the point with incresing the diversity.

In [8]:
import gc
import math
import os
from pathlib import Path
import bitsandbytes as bnb
from transformers import AutoTokenizer
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DiffusionPipeline,
    UNet2DConditionModel,
)

from transformers import CLIPTextModel

class DreamBoothDataset(Dataset):
    def __init__(
        self,
        instance_data_root,
        instance_prompt,
        tokenizer,
        class_data_root=None,
        class_prompt=None,
        class_num=None,
        size=512,
        tokenizer_max_length=None,
    ):
        self.size = size
        self.tokenizer = tokenizer
        self.tokenizer_max_length = tokenizer_max_length

        self.instance_data_root = Path(instance_data_root)
        if not self.instance_data_root.exists():
            raise ValueError(f"Instance {self.instance_data_root} images root doesn't exists.")

        self.instance_images_path = list(Path(instance_data_root).iterdir())
        self.num_instance_images = len(self.instance_images_path)
        self.instance_prompt = instance_prompt
        self._length = self.num_instance_images
        self.class_data_root = None
        if class_data_root is not None:
            self.class_data_root = Path(class_data_root)
            print(self.class_data_root)
            self.class_images_path = list(self.class_data_root.iterdir())
            print(self.class_images_path)
            self.num_class_images = min(len(self.class_images_path), class_num)
            self._length = max(self.num_class_images, self.num_instance_images)
            self.class_prompt = class_prompt


        self.image_transforms = transforms.Compose(
            [
                transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.RandomCrop(size),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self):
        return self._length

    def __getitem__(self, index):
        example = {}
        instance_image = Image.open(self.instance_images_path[index % self.num_instance_images])
        instance_image = exif_transpose(instance_image)

        example["instance_images"] = self.image_transforms(instance_image)

        text_inputs = self.tokenizer(self.instance_prompt,
                                     truncation=True,
                                     padding="max_length",
                                     max_length=self.tokenizer_max_length,
                                     return_tensors='pt')
        example["instance_prompt_ids"] = text_inputs.input_ids
        example["instance_attention_mask"] = text_inputs.attention_mask

        if self.class_data_root:
            class_image = Image.open(self.class_images_path[index % self.num_class_images])
            class_image = exif_transpose(class_image)
            example["class_images"] = self.image_transforms(class_image)

            class_text_inputs = self.tokenizer(self.class_prompt,
                                     truncation=True,
                                     padding="max_length",
                                     max_length=self.tokenizer_max_length,
                                     return_tensors='pt')

            example["class_prompt_ids"] = class_text_inputs.input_ids
            example["class_attention_mask"] = class_text_inputs.attention_mask

        return example


Parameter `with_prior_preservation` should only be set to True in the increasing the diversity section.

In [9]:
def collate_fn(examples, with_prior_preservation=False):
    input_ids = [example["instance_prompt_ids"] for example in examples]
    pixel_values = [example["instance_images"] for example in examples]

    attention_mask = [example["instance_attention_mask"] for example in examples]

    if with_prior_preservation:
        input_ids += [example["class_prompt_ids"] for example in examples]
        pixel_values += [example["class_images"] for example in examples]

        attention_mask += [example["class_attention_mask"] for example in examples]

    pixel_values = torch.stack(pixel_values)
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()

    input_ids = torch.cat(input_ids, dim=0)

    batch = {
        "input_ids": input_ids,
        "pixel_values": pixel_values,
    }

    attention_mask = torch.cat(attention_mask, dim=0)
    batch["attention_mask"] = attention_mask

    return batch


## Implement training and finetune the model [4pt]
You should:
*       Only train the unet component
*       Use gradient checkpointing
*       Use 8-bit optimizers, such as `bnb.optim.AdamW8bit`

Your code should run on a T4 GPU on Colab.

In [10]:
from bitsandbytes.optim import AdamW8bit
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
import plotly.express as px
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
def get_components(path, device):
    # Load the model components
    # from the provided path
    # Choose the components for training
    # Be sure to move them to the correct device
    ### CODE START ###

    pipeline = StableDiffusionPipeline.from_pretrained(path)
    pipeline = pipeline.to(device)

    text_encoder = pipeline.text_encoder.to(device)
    vae = pipeline.vae.to(device)
    unet = pipeline.unet.to(device)

    noise_scheduler = pipeline.scheduler
    tokenizer = pipeline.tokenizer

    ### CODE END ###
    return noise_scheduler, text_encoder, vae, unet, tokenizer

Implement the diffusion model objective for finetuning.

In [12]:
def train(
    output_dir,
    model_path,
    noise_scheduler,
    text_encoder,
    vae,
    unet,
    optimizer,
    dataloader,
    train_steps,
    prior_preservation
):

    ### CODE START ###

    unet.train()
    text_encoder.eval()
    vae.eval()

    enable_grad = [(text_encoder, False), (vae, False), (unet,True)]
    for elem, enable in enable_grad:
        for param in elem.parameters():
            param.requires_grad = enable

    num_train_epochs = train_steps // len(dataloader)
    print(f'{len(dataloader)=}')

    scaling_factor = 0.18215
    scaler = GradScaler()
    losses = []

    for epoch in range(num_train_epochs):
        epoch_loss = 0
        for step, batch in enumerate(dataloader):
            optimizer.zero_grad()

            zipped = zip(
                batch['pixel_values'],
                batch['input_ids'],
                batch['attention_mask'],
            )
            loss = 0

            for pixel_values, input_ids, attention_mask in zipped:
                pixel_values = pixel_values.unsqueeze(0).to(device)
                input_ids = input_ids.unsqueeze(0).to(device)
                attention_mask = attention_mask.unsqueeze(0).to(device)

                # Map the pixel values into the latent space by using the Variational Autoencoder
                with torch.no_grad():
                    latents = vae.encode(pixel_values).latent_dist.sample()
                # The encoded inputs need to be scaled using the scaling factor (which is 0.18215 for the VAE)
                    latents *= scaling_factor

                # Sample random gaussian noise of the same shape as the model input
                noise = torch.randn_like(latents, device = device)

                # Sample random timesteps for each image in batch
                timesteps = torch.randint(
                    low=0,
                    high=noise_scheduler.config.num_train_timesteps,
                    size=(latents.shape[0],),
                    dtype=torch.long,
                    device=device
                )

                # Use the scheduler to add the correct amount of noise to the model input
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # Encode the text prompts using the text encoder
                with torch.no_grad():
                    text_embeddings = text_encoder(
                        input_ids,
                        attention_mask = attention_mask,
                    ).last_hidden_state

                # Use the unet to predict the noise (so your network should be able to predict the noise that can be substracted from the noisy image)
                predicted_noise = unet(
                    noisy_latents,
                    timesteps,
                    text_embeddings,
                ).sample

                # Calculate the loss
                loss += torch.nn.functional.mse_loss(predicted_noise, noise)

            # loss.backward()
            scaler.scale(loss).backward()

            # Update the model
            # optimizer.step()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

        losses += [epoch_loss]
        print(f"\n Epoch {epoch}: loss : {epoch_loss}")

    #  Save the newly trained pipeline
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, f"unet_trained.pt")
        torch.save(unet.state_dict(), path),

    # Clear the gpu memory
    del unet
    del vae
    del text_encoder
    del dataloader
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    fig = px.line(y=losses, labels={'y':'loss'})
    return fig
    ### CODE END ###

Run the training and perform the evaluation. You can use the suggested hyperparameters, but you can also change them if that would make your model better.
Evaluating generative models is difficult and calculating the metrics generally requires generating thousands of images, therefore
your homework will be graded based on the visual results. In this low-GPU memory setup, the expected result is a model, that will
generate photos of a dog closely resembling the training data, while allowing for some additions - like generating the dog on a skate
board etc.

In [13]:
%%script false --no-raise-error
model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 5e-6
learning_rate = 3e-6
instance_data_dir = '/content/dog'
instance_prompt = "a photo of sks dog"
resolution = 512
batch_size = 1
output_dir = '/content/trained_model'
train_steps = 400 * 5


### CODE START ###
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

(noise_scheduler, text_encoder, vae, unet, tokenizer) = \
    get_components(model_path, device)

optimizer = AdamW8bit(unet.parameters(), lr=learning_rate)

instance_data_dir = 'data/dog'
dataset = DreamBoothDataset(
    instance_data_dir,
    instance_prompt,
    tokenizer,
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle = True,
    collate_fn=collate_fn,
)

output_dir = 'data/output'
fig = train(
    output_dir,
    model_path,
    noise_scheduler,
    text_encoder,
    vae,
    unet,
    optimizer,
    dataloader,
    train_steps=train_steps,
    prior_preservation=False,
)

### CODE END ###

In [14]:
%%script false --no-raise-error
def generate_images_custom_unet(prompt, num_images, model_path, data_path, device, batch_size):
    ### CODE START ###

    pipe = StableDiffusionPipeline.from_pretrained(model_path)
    pipe = pipe.to(device)

    custom_unet_path = os.path.join(data_path, 'output/unet_trained.pt')
    pipe.unet.load_state_dict(torch.load(custom_unet_path, weights_only=False, map_location=device))
    pipe.unet = pipe.unet.to(device)
    pipe.unet.eval()

    total_batches = (num_images + batch_size - 1) // batch_size
    image_count = 0

    for batch in tqdm(range(total_batches), desc=prompt, disable=False):
        current_batch_size = min(batch_size, num_images - image_count)
        images = pipe([prompt] * current_batch_size).images

        for img in images:
            save_path = os.path.join(data_path, 'output_images')
            filename = f"{save_path}/{prompt.replace(' ', '_')}_{image_count}.png"
            img.save(filename)
            image_count += 1

    ### CODE END ###


device = 'cuda' if torch.cuda.is_available() else 'cpu'
prompts = ["a photo of sks dog", "a sailboat at sea", "a child playing", "a photo of a cat", "a photo of a dog"]
num_images = 3
sample_batch_size = 3
model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
data_path = "/home/nojak/Git/ML12/VR/hw3/data"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

for prompt in prompts:
    generate_images_custom_unet(prompt, num_images, model_path, data_path, device, sample_batch_size)

## Increasing image diversity [2pt]
Looking at the images you generated using your finetuned model and comparing them with ones from the original model, you can likely notice the issues with the diversity of generated images, especially animal images.
Your task will be to consider two approaches that might allow for improving that.

### Prior preservation [1pt]
This is a classical method for counteracting forgetting during finetuning, and this is described in the DreamBooth paper. In addition to instance prompt "A photo of sks dog", a class prompt "A photo of a dog" is defined. You should generate 50 images for the class prompt from the original pretrained model. Then, during the finetuning phase, your model at each gradient update should receive a gradient from one image from the original class, as well as one instance image.

### Mixing models in generation [1pt]
During denoising, you can perform certain steps of the denoising using the finetuned model and other using the original one. Try to either perform a suffix or a prefix of steps with the original model, report any interesting findings in the report. Report which scheduler are you using as well as the number of steps. We want a configuration, that for a prompt "A photo of sks dog" works similarily to the finetuned model and for a prompt "A photo of a dog" works similarily to the original model. Such a configuration does not exist, but some interesting questions are:
- How using the original model at a prefix differs from using it at the suffix
- What is the best configuration you find
- Denote the original model by $O$ and finetuned by $F$, assume you denoise for $k$ steps, let $N$ be noise (a random variable from a normal distribution), $P$ be a prompt and $N_m^{k-m}=F^{k-m}O^mN$ be the random variable after $m$ steps of denoising with the original model and $k-m$ with the finetuned one, conditioned on the prompt.  How does the distribution of the random variable $N_m^{k-m}$ change with $m$ for both $P=$'A photo of sks dog' and $P=$'A photo of a dog'? Is the change very 'smooth' and gradual or are there some abrupt changes? Note, that this is not a formal mathematical problem, and you are expected to investigate the distributions by sampling a few images for the same noise for different values of $m$ and looking at them. (the same goes for when we use the original model for a prefix)

In [15]:
%%script false --no-raise-error
prompt = "a photo of a dog"
num_images = 50
sample_batch_size = 5
model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
save_path = "data/dog_images"
generate_images(prompt, num_images, model_path, save_path, device, sample_batch_size, precision = 16)

In [16]:
%%script false --no-raise-error
import os
import zipfile

image_dir = 'data/dog_images'
zip_filename = 'dog_images.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for filename in os.listdir(image_dir):
        file_path = os.path.join(image_dir, filename)
        if os.path.isfile(file_path):
            zipf.write(file_path, arcname=filename)

print(f"Zipped images saved to {zip_filename}")


In [ ]:
## Prior preservation

model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 3e-6
instance_data_dir = 'data/dog'
instance_prompt = "a photo of sks dog"
class_data_dir = 'data/dog_images'
class_prompt = "a photo of a dog"
resolution = 512
batch_size = 1
output_dir = 'data/trained_model_prior'
train_steps = 400 * 5


### CODE START ###
(noise_scheduler, text_encoder, vae, unet, tokenizer) = \
    get_components(model_path, device)

optimizer = AdamW8bit(unet.parameters(), lr=learning_rate)

dataset = DreamBoothDataset(
    instance_data_dir,
    instance_prompt,
    tokenizer,
    class_data_dir,
    class_prompt,
    class_num=50,
)

prior_collate = lambda x: collate_fn(x, True)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=prior_collate,
)

fig = train(
    output_dir,
    model_path,
    noise_scheduler,
    text_encoder,
    vae,
    unet,
    optimizer,
    dataloader,
    train_steps=train_steps,
    prior_preservation=True,
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

data/dog_images
[PosixPath('data/dog_images/a_photo_of_a_dog_3.png'), PosixPath('data/dog_images/a_photo_of_a_dog_6.png'), PosixPath('data/dog_images/a_photo_of_a_dog_25.png'), PosixPath('data/dog_images/a_photo_of_a_dog_20.png'), PosixPath('data/dog_images/a_photo_of_a_dog_45.png'), PosixPath('data/dog_images/a_photo_of_a_dog_28.png'), PosixPath('data/dog_images/a_photo_of_a_dog_41.png'), PosixPath('data/dog_images/a_photo_of_a_dog_1.png'), PosixPath('data/dog_images/a_photo_of_a_dog_19.png'), PosixPath('data/dog_images/a_photo_of_a_dog_9.png'), PosixPath('data/dog_images/a_photo_of_a_dog_46.png'), PosixPath('data/dog_images/a_photo_of_a_dog_43.png'), PosixPath('data/dog_images/a_photo_of_a_dog_38.png'), PosixPath('data/dog_images/a_photo_of_a_dog_29.png'), PosixPath('data/dog_images/a_photo_of_a_dog_7.png'), PosixPath('data/dog_images/a_photo_of_a_dog_33.png'), PosixPath('data/dog_images/a_photo_of_a_dog_44.png'), PosixPath('data/dog_images/a_photo_of_a_dog_10.png'), PosixPath('data/

In [ ]:
### CODE START ###

### CODE END ###

In [ ]:
## Mixing models

from PIL import Image

@torch.no_grad()
def generate_mixed_denoising_images(prompt, num_images, m, mix_mode, model_path, data_path, device, num_inference_steps=50):
    """
    m: number of steps to run with the original model
    mix_mode: "prefix" or "suffix"
    """

    original_pipe = StableDiffusionPipeline.from_pretrained(model_path).to(device)
    finetuned_pipe = StableDiffusionPipeline.from_pretrained(model_path).to(device)

    custom_unet_path = os.path.join(data_path, 'output/unet_trained.pt')
    finetuned_pipe.unet.load_state_dict(torch.load(custom_unet_path, weights_only=False, map_location=device))
    finetuned_pipe.unet = finetuned_pipe.unet.to(device).eval()

    scheduler = DDIMScheduler.from_pretrained(model_path, subfolder="scheduler")
    scheduler.set_timesteps(num_inference_steps)

    text_input = original_pipe.tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    text_embeddings = original_pipe.text_encoder(text_input)[0]

    for i in range(num_images):
        # Sample initial noise
        latents = torch.randn((1, 4, 64, 64), device=device, dtype=torch.float16)
        latents = latents * scheduler.init_noise_sigma

        for step_index, t in enumerate(scheduler.timesteps):
            use_finetuned = (mix_mode == "prefix" and step_index >= m) or \
                (mix_mode == "suffix" and step_index < num_inference_steps - m)

            unet = finetuned_pipe.unet if use_finetuned else original_pipe.unet

            # Predict noise residual
            latent_model_input = latents
            latent_model_input = scheduler.scale_model_input(latent_model_input, t)
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

            # Denoising step
            latents = scheduler.step(noise_pred, t, latents).prev_sample

        # Decode the final latents
        image = original_pipe.decode_latents(latents)
        image = original_pipe.numpy_to_pil(image)[0]

        # Save
        save_path = os.path.join(data_path, 'mixed_outputs')
        os.makedirs(save_path, exist_ok=True)
        filename = f"{prompt.replace(' ', '_')}_m{m}_{mix_mode}_{i}.png"
        image.save(os.path.join(save_path, filename))


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
prompts = ["a photo of sks dog", "a photo of a dog"]
model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
data_path = "/home/nojak/Git/ML12/VR/hw3/data"

for prompt in prompts:
    for m in [10, 20, 30, 40]:
        for mode in ['prefix', 'suffix']:
            generate_mixed_denoising_images(
                prompt,
                num_images=3,
                m=m,
                mix_mode=mode,
                model_path=model_path,
                data_path=data_path,
                device=device,
            )